# Explainable Hybrid Forecasting of Residential Energy Consumption
## with Demographic Fairness Evaluation

**Dataset:** Low Carbon London smart meter data (3 blocks, ACORN groups embedded)  
**Models:** Prophet + LSTM → Ridge meta-learner → **ONE hybrid model**  
**Evaluation:** SHAP attribution + demographic fairness audit (Kruskal-Wallis)

---

### ⚠️ Key methodological point

The fairness audit uses **a single hybrid model trained on pooled data from all three ACORN
groups**. Its errors are then **disaggregated by the ACORN group of each household**.

This is deliberate. It answers RQ2 — *does one deployed model serve all households equitably?* —
which is the situation that arises in practice when a utility trains one model and applies it
across its whole customer base. Fitting a **separate model per group** would answer a different
question (whether each group is intrinsically harder to forecast) and would not match RQ2.

---

### Getting the real data

1. Download from the London Datastore:
   https://data.london.gov.uk/dataset/smartmeter-energy-use-data-in-london-households
2. You need `informations_households.csv` and three block files from `halfhourly_dataset/`:
   `block_0.csv`, `block_44.csv`, `block_75.csv`
3. Place them so the paths match `DATA_DIR` below, then set `USE_SYNTHETIC = False`.

A synthetic sample in the **exact same schema and block structure** ships with this notebook, so
the pipeline runs immediately.


## 0. Install Dependencies

In [ ]:
import subprocess, sys
for pkg in ["pandas","numpy","matplotlib","seaborn","prophet","tensorflow",
            "scikit-learn","shap","scipy","scikit-posthocs"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("Packages installed.")


## 1. Configuration

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import os

# ─────────────────────────────────────────────────────────────────────
USE_SYNTHETIC = False          # ← set False once you have the real files
DATA_DIR      = "."   # real block files sit in the same folder as this notebook  # folder holding informations_households.csv
                              # and halfhourly_dataset/
BLOCKS        = ["block_0", "block_44", "block_75"]

SEQ_LEN     = 48    # LSTM look-back (48 hours = 2 diurnal cycles)
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10  # test = remaining 10%
# meta-learner is calibrated on the validation partition (see section 8)
SEED        = 42
MAX_LSTM_SEQ = 40_000   # cap training sequences for tractability
# ─────────────────────────────────────────────────────────────────────

np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
GROUPS = ['Affluent', 'Comfortable', 'Adversity']
C = {'Affluent':'#2E75B6', 'Comfortable':'#ED9B40', 'Adversity':'#C0392B',
     'prophet':'#ED9B40', 'lstm':'#7D3C98', 'hybrid':'#27AE60', 'actual':'#212121'}
print("Config set. USE_SYNTHETIC =", USE_SYNTHETIC)


## 2. Load and Merge (inner join on LCLid)

In [ ]:
def load_blocks(data_dir, blocks):
    info = pd.read_csv(os.path.join(data_dir, "informations_households.csv"))

    frames = []
    for b in blocks:
        path = os.path.join(data_dir, "halfhourly_dataset", f"{b}.csv")
        if not os.path.exists(path):
            path = os.path.join(data_dir, f"{b}.csv")   # fall back to files directly in DATA_DIR
        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]
        ts = 'tstp' if 'tstp' in df.columns else 'DateTime'
        en = [c for c in df.columns if 'energy' in c.lower() or 'kwh' in c.lower()][0]
        df = df.rename(columns={ts: 'datetime', en: 'energy'})
        frames.append(df[['LCLid', 'datetime', 'energy']])
    readings = pd.concat(frames, ignore_index=True)

    # 'Null' -> NaN ; parse timestamps
    readings['energy']   = pd.to_numeric(readings['energy'], errors='coerce')
    readings['datetime'] = pd.to_datetime(readings['datetime'], errors='coerce')
    readings = readings.dropna(subset=['datetime'])

    # INNER JOIN -> keeps only households with a valid ACORN group
    valid = info[info['Acorn_grouped'].isin(GROUPS)]
    merged = readings.merge(valid[['LCLid','Acorn_grouped']], on='LCLid', how='inner')
    return merged, valid


data, info = load_blocks(DATA_DIR, BLOCKS)

print(f"Households retained : {data['LCLid'].nunique()}")
print(f"Half-hourly readings: {len(data):,}")
print(f"Missing ('Null')    : {data['energy'].isna().sum():,} "
      f"({100*data['energy'].isna().mean():.2f}%)")
print("\nHouseholds per ACORN group:")
print(info['Acorn_grouped'].value_counts().reindex(GROUPS).to_string())


## 3. Preprocess: interpolate ≤ 4 half-hours, aggregate to hourly

In [ ]:
data = data.sort_values(['LCLid','datetime'])

# Step 3: interpolate short gaps only (max 4 consecutive half-hours = 2 hours)
data['energy'] = data.groupby('LCLid')['energy'].transform(
    lambda s: s.interpolate(limit=4, limit_area='inside'))

# Step 4: aggregate half-hourly -> hourly mean, per household
hourly = (data.set_index('datetime')
              .groupby('LCLid')['energy']
              .resample('h').mean()
              .reset_index())
hourly = hourly.merge(info[['LCLid','Acorn_grouped']], on='LCLid', how='left')
hourly = hourly.dropna(subset=['energy'])

print(f"Hourly rows: {len(hourly):,}")
print(f"Period     : {hourly['datetime'].min()}  →  {hourly['datetime'].max()}")
print("\nHourly observations per group:")
print(hourly.groupby('Acorn_grouped')['energy'].agg(['count','mean']).reindex(GROUPS).round(4).to_string())


## 4. Chronological Split (no shuffling)

In [ ]:
times = np.sort(hourly['datetime'].unique())
n_t   = len(times)
i_tr  = int(n_t * TRAIN_RATIO)
i_va  = int(n_t * (TRAIN_RATIO + VAL_RATIO))

t_train_end = times[i_tr - 1]
t_val_end   = times[i_va - 1]

train = hourly[hourly['datetime'] <= t_train_end]
val   = hourly[(hourly['datetime'] > t_train_end) & (hourly['datetime'] <= t_val_end)]
test  = hourly[hourly['datetime'] > t_val_end]

print(f"Train: {len(train):,} rows  ({train['datetime'].min().date()} → {train['datetime'].max().date()})")
print(f"Val  : {len(val):,} rows  ({val['datetime'].min().date()} → {val['datetime'].max().date()})")
print(f"Test : {len(test):,} rows  ({test['datetime'].min().date()} → {test['datetime'].max().date()})")


## 5. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 8))
fig.suptitle('Low Carbon London — Exploratory Analysis', fontsize=14, fontweight='bold')

# daily mean by group
a = ax[0,0]
for g in GROUPS:
    s = (hourly[hourly['Acorn_grouped']==g]
         .set_index('datetime')['energy'].resample('D').mean())
    a.plot(s.index, s.values, label=g, color=C[g], lw=1.1)
a.set_title('Daily mean consumption by ACORN group'); a.set_ylabel('kWh/h'); a.legend()

# daily profile
a = ax[0,1]
for g in GROUPS:
    sub = hourly[hourly['Acorn_grouped']==g].copy()
    sub['hour'] = sub['datetime'].dt.hour
    p = sub.groupby('hour')['energy'].mean()
    a.plot(p.index, p.values, label=g, color=C[g], lw=2, marker='o', ms=3)
a.set_title('Average daily profile'); a.set_xlabel('Hour'); a.set_ylabel('kWh/h'); a.legend()

# distribution
a = ax[1,0]
for g in GROUPS:
    a.hist(hourly[hourly['Acorn_grouped']==g]['energy'], bins=60, alpha=0.5,
           label=g, color=C[g], density=True)
a.set_title('Consumption distribution'); a.set_xlabel('kWh/h'); a.legend()

# variability (std) — a proxy for forecasting difficulty
a = ax[1,1]
v = hourly.groupby(['Acorn_grouped','LCLid'])['energy'].std().groupby('Acorn_grouped').mean().reindex(GROUPS)
bars = a.bar(v.index, v.values, color=[C[g] for g in v.index], alpha=0.85, edgecolor='white')
a.set_title('Mean within-household variability (std)'); a.set_ylabel('kWh/h')
for b_, val in zip(bars, v.values):
    a.text(b_.get_x()+b_.get_width()/2, val+0.002, f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout(); plt.savefig('fig_eda.png', dpi=150, bbox_inches='tight'); plt.show()


## 6. Base Model 1 — Prophet (fitted ONCE)

Prophet is fitted on the **pooled composite series** (mean hourly consumption across all
households). For each household, the Prophet feature is the composite prediction rescaled by
that household's own training-period level, so the feature is on the right scale for each
household while still coming from **one fitted Prophet model**.


In [ ]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mae(y, p):  return mean_absolute_error(y, p)
def rmse(y, p): return float(np.sqrt(mean_squared_error(y, p)))

# composite (pooled) series
composite = hourly.groupby('datetime')['energy'].mean().sort_index()
comp_train = composite[composite.index <= t_train_end]

dfp = comp_train.reset_index(); dfp.columns = ['ds','y']

prophet = Prophet(daily_seasonality=True, weekly_seasonality=True,
                  yearly_seasonality=False, changepoint_prior_scale=0.05)
prophet.fit(dfp)

# predict across the whole span
full = pd.DataFrame({'ds': composite.index})
fc   = prophet.predict(full)
prophet_curve = pd.Series(fc['yhat'].values, index=composite.index)

# household level factors (from TRAINING period only — no leakage)
hh_level = train.groupby('LCLid')['energy'].mean()
comp_level = comp_train.mean()
scale = (hh_level / comp_level).rename('scale')

print(f"Prophet fitted on {len(dfp):,} composite hourly points.")
print(f"Household scale factors: min={scale.min():.2f}, median={scale.median():.2f}, max={scale.max():.2f}")

fig, a = plt.subplots(figsize=(14,3.5))
a.plot(composite.index, composite.values, color=C['actual'], lw=0.8, label='Composite actual')
a.plot(prophet_curve.index, prophet_curve.values, color=C['prophet'], lw=1.2, ls='--', label='Prophet')
a.axvline(t_val_end, color='red', ls=':', lw=1.5, label='Test starts')
a.set_title('Prophet on the pooled composite series'); a.legend(); a.set_ylabel('kWh/h')
plt.tight_layout(); plt.savefig('fig_prophet.png', dpi=150, bbox_inches='tight'); plt.show()


## 7. Base Model 2 — LSTM (trained ONCE on pooled sequences)

Sequences are drawn from **every household** across all three ACORN groups, so a single LSTM
learns from the pooled population.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
tf.random.set_seed(SEED)

# wide matrix: rows = timestamps, cols = households
wide = hourly.pivot_table(index='datetime', columns='LCLid', values='energy')
wide = wide.sort_index().interpolate(limit=2, limit_area='inside')
households = list(wide.columns)
print(f"Wide matrix: {wide.shape[0]:,} timestamps × {wide.shape[1]} households")

# scale on TRAINING data only
scaler = MinMaxScaler()
scaler.fit(wide.loc[wide.index <= t_train_end].values.reshape(-1,1))
wide_s = pd.DataFrame(
    scaler.transform(wide.values.reshape(-1,1)).reshape(wide.shape),
    index=wide.index, columns=wide.columns)

from numpy.lib.stride_tricks import sliding_window_view

def build_sequences(w, t_start, t_end, seq_len):
    """Vectorised. Returns X, y, and the household + timestamp of every sample."""
    arr = w.values                                   # (T, H)
    idx = w.index
    T   = len(idx)

    win = sliding_window_view(arr, (seq_len,), axis=0)   # (T-seq+1, H, seq)

    positions = np.arange(seq_len, T)
    tt = idx[positions]
    mask = (tt > t_start) & (tt <= t_end) if t_start is not None else (tt <= t_end)
    pos = positions[mask]

    Xw = win[pos - seq_len]        # (n, H, seq)  window ending right before target
    Y  = arr[pos]                  # (n, H)
    n, H, S = Xw.shape

    X  = Xw.reshape(n * H, S)
    y  = Y.reshape(n * H)
    hh = np.tile(np.asarray(w.columns), n)
    ts = np.repeat(idx[pos].values, H)

    ok = (~np.isnan(X).any(axis=1)) & (~np.isnan(y))
    return X[ok], y[ok], hh[ok], ts[ok]

Xtr, ytr, _, _   = build_sequences(wide_s, None,        t_train_end, SEQ_LEN)
Xva, yva, hh_va, ts_va = build_sequences(wide_s, t_train_end, t_val_end, SEQ_LEN)
Xte, yte, hh_te, ts_te = build_sequences(wide_s, t_val_end, wide.index.max(), SEQ_LEN)

# cap training size for tractability
if len(Xtr) > MAX_LSTM_SEQ:
    sel = np.random.choice(len(Xtr), MAX_LSTM_SEQ, replace=False)
    Xtr, ytr = Xtr[sel], ytr[sel]

Xtr = Xtr.reshape(-1, SEQ_LEN, 1); Xva = Xva.reshape(-1, SEQ_LEN, 1); Xte = Xte.reshape(-1, SEQ_LEN, 1)
print(f"Train seqs {Xtr.shape} | Val {Xva.shape} | Test {Xte.shape}")

lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LEN,1)), Dropout(0.2),
    LSTM(32), Dropout(0.2),
    Dense(16, activation='relu'), Dense(1)
])
lstm.compile(optimizer='adam', loss='mse')
hist = lstm.fit(Xtr, ytr, validation_data=(Xva, yva),
                epochs=10, batch_size=256, verbose=1,
                callbacks=[EarlyStopping(patience=4, restore_best_weights=True)])


## 7b. Look-back window selection (24 / 48 / 72 hours)

A short experiment comparing validation loss across candidate look-back windows, to confirm the 48-hour choice used above. Trains a small LSTM on a capped sample for each window and reports the best validation MSE. Optional; set `RUN_WINDOW_EXPERIMENT = True` to execute.

In [ ]:
RUN_WINDOW_EXPERIMENT = False   # set True to reproduce the window-selection experiment

if RUN_WINDOW_EXPERIMENT:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    win_scores = {}
    for W in [24, 48, 72]:
        Xtr_w, ytr_w, _, _ = build_sequences(wide_s, None, t_train_end, W)
        Xva_w, yva_w, _, _ = build_sequences(wide_s, t_train_end, t_val_end, W)
        if len(Xtr_w) > 15000:
            sel = np.random.choice(len(Xtr_w), 15000, replace=False); Xtr_w, ytr_w = Xtr_w[sel], ytr_w[sel]
        if len(Xva_w) > 15000:
            sel = np.random.choice(len(Xva_w), 15000, replace=False); Xva_w, yva_w = Xva_w[sel], yva_w[sel]
        Xtr_w = Xtr_w.reshape(-1, W, 1); Xva_w = Xva_w.reshape(-1, W, 1)
        mw = Sequential([LSTM(64, return_sequences=True, input_shape=(W,1)), Dropout(0.2),
                         LSTM(32), Dropout(0.2), Dense(16, activation='relu'), Dense(1)])
        mw.compile(optimizer='adam', loss='mse')
        h = mw.fit(Xtr_w, ytr_w, validation_data=(Xva_w, yva_w), epochs=4, batch_size=512, verbose=0)
        win_scores[W] = float(min(h.history['val_loss']))
        print(f"look-back {W:>3}h  best validation MSE = {win_scores[W]:.5f}")
    best_W = min(win_scores, key=win_scores.get)
    print(f"\nSelected look-back window: {best_W} hours")
else:
    print("Window experiment skipped (SEQ_LEN =", SEQ_LEN, "used). Set RUN_WINDOW_EXPERIMENT = True to run it.")


## 8. Assemble the SINGLE Hybrid Model (Ridge meta-learner)

The Ridge meta-learner is calibrated on the **validation** partition, using the Prophet and LSTM predictions generated there as its two input features, and is then evaluated on the fully held-out **test** partition. Calibrating on validation rather than on any part of the test set keeps the evaluation strictly out-of-sample.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

inv = lambda a: scaler.inverse_transform(np.asarray(a).reshape(-1,1)).flatten()

def prophet_feature(hh, ts):
    # one fitted Prophet model, rescaled per household
    return np.array([prophet_curve.loc[t] * scale.get(h, 1.0) for h, t in zip(hh, ts)])

# --- base-model predictions on the VALIDATION partition (used to calibrate the meta-learner) ---
val_lstm    = inv(lstm.predict(Xva, verbose=0).flatten())
val_actual  = inv(yva)
val_prophet = prophet_feature(hh_va, ts_va)
V = pd.DataFrame({'LCLid': hh_va, 'datetime': ts_va,
                  'actual': val_actual, 'prophet': val_prophet, 'lstm': val_lstm}).dropna(subset=['prophet','lstm','actual'])

# --- base-model predictions on the held-out TEST partition (final evaluation) ---
lstm_pred    = inv(lstm.predict(Xte, verbose=0).flatten())
actual       = inv(yte)
prophet_pred = prophet_feature(hh_te, ts_te)
P = pd.DataFrame({'LCLid': hh_te, 'datetime': ts_te,
                  'actual': actual, 'prophet': prophet_pred, 'lstm': lstm_pred}
                ).merge(info[['LCLid','Acorn_grouped']], on='LCLid', how='left').dropna(subset=['prophet','lstm','actual'])

# --- meta-learner: fit on VALIDATION only, so the test set stays fully held out ---
ms = StandardScaler()
Xc = ms.fit_transform(V[['prophet','lstm']].values)     # calibration features (validation)
Xf = ms.transform(P[['prophet','lstm']].values)         # evaluation features (test)
meta = Ridge(alpha=1.0)
meta.fit(Xc, V['actual'].values)
final = P.copy()
final['hybrid'] = meta.predict(Xf)

print(f"Meta-learner calibrated on {len(V):,} validation rows; evaluated on {len(final):,} held-out test rows.")
print(f"Ridge coefficients — Prophet: {meta.coef_[0]:.4f}, LSTM: {meta.coef_[1]:.4f}")

results = pd.DataFrame({
    'Model':  ['Prophet','LSTM','Hybrid'],
    'MAE':  [mae(final['actual'], final['prophet']),
             mae(final['actual'], final['lstm']),
             mae(final['actual'], final['hybrid'])],
    'RMSE': [rmse(final['actual'], final['prophet']),
             rmse(final['actual'], final['lstm']),
             rmse(final['actual'], final['hybrid'])],
})
print("\n── RQ1: Model comparison (held-out test set) ──")
print(results.round(4).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15,4))

# one household's forecast
h0 = final['LCLid'].iloc[0]
s  = final[final['LCLid']==h0].sort_values('datetime')
ax[0].plot(s['datetime'], s['actual'], color=C['actual'], lw=1.2, label='Actual')
ax[0].plot(s['datetime'], s['hybrid'], color=C['hybrid'], lw=1.2, ls='--', label='Hybrid')
ax[0].set_title(f'Hybrid forecast — household {h0}'); ax[0].legend(); ax[0].set_ylabel('kWh/h')
ax[0].tick_params(axis='x', rotation=30)

x = np.arange(3); w = 0.35
ax[1].bar(x-w/2, results['MAE'],  w, label='MAE',  color=C['prophet'], alpha=0.9)
ax[1].bar(x+w/2, results['RMSE'], w, label='RMSE', color=C['lstm'],    alpha=0.9)
ax[1].set_xticks(x); ax[1].set_xticklabels(results['Model'])
ax[1].set_title('Model comparison'); ax[1].set_ylabel('Error (kWh/h)'); ax[1].legend()

plt.tight_layout(); plt.savefig('fig_models.png', dpi=150, bbox_inches='tight'); plt.show()


## 9. RQ3 — SHAP Attribution on the Hybrid Meta-Learner

The meta-learner's two input features **are** the Prophet and LSTM predictions, so SHAP values
tell us how much the hybrid relies on each base model. We also disaggregate the attributions by
ACORN group.


In [ ]:
import shap

bg  = shap.kmeans(Xc, 30)
ex  = shap.KernelExplainer(meta.predict, bg)

samp = final.sample(min(150, len(final)), random_state=SEED).copy()
Xs   = ms.transform(samp[['prophet','lstm']].values)
print("Computing SHAP values...")
sv   = ex.shap_values(Xs, nsamples=60)
FNAMES = ['Prophet prediction', 'LSTM prediction']

print("\nMean |SHAP| (overall):")
for f, v in zip(FNAMES, np.abs(sv).mean(axis=0)):
    print(f"  {f:<20} {v:.4f}")

shap.summary_plot(sv, Xs, feature_names=FNAMES, plot_type='bar', show=False)
plt.title('SHAP feature importance — hybrid meta-learner')
plt.tight_layout(); plt.savefig('fig_shap_bar.png', dpi=150, bbox_inches='tight'); plt.show()

shap.summary_plot(sv, Xs, feature_names=FNAMES, show=False)
plt.title('SHAP beeswarm')
plt.tight_layout(); plt.savefig('fig_shap_beeswarm.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# SHAP attribution disaggregated by ACORN group
samp['shap_prophet'] = sv[:,0]
samp['shap_lstm']    = sv[:,1]

attr = samp.groupby('Acorn_grouped')[['shap_prophet','shap_lstm']].apply(
    lambda d: pd.Series({'Prophet': d['shap_prophet'].abs().mean(),
                         'LSTM':    d['shap_lstm'].abs().mean()})
).reindex(GROUPS)

print("Mean |SHAP| by ACORN group:")
print(attr.round(4).to_string())

fig, a = plt.subplots(figsize=(7,4))
attr.plot(kind='bar', ax=a, color=[C['prophet'], C['lstm']], alpha=0.9, edgecolor='white')
a.set_title('Does the hybrid rely on its base models differently by group?')
a.set_ylabel('Mean |SHAP|'); a.set_xlabel(''); a.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.savefig('fig_shap_by_group.png', dpi=150, bbox_inches='tight'); plt.show()


## 10. RQ2 — Demographic Fairness Audit

**One hybrid model. Its errors, split by the ACORN group of each household.**


In [ ]:
from scipy.stats import kruskal

final['abs_error'] = (final['actual'] - final['hybrid']).abs()

# per-group error metrics
audit = final.groupby('Acorn_grouped').apply(
    lambda d: pd.Series({
        'Households': d['LCLid'].nunique(),
        'N predictions': len(d),
        'MAE':  d['abs_error'].mean(),
        'RMSE': float(np.sqrt((d['abs_error']**2).mean())),
    })
).reindex(GROUPS)

print("── Fairness audit: hybrid model error by ACORN group ──")
print(audit.round(4).to_string())

# Kruskal-Wallis across the three groups
arrays = [final.loc[final['Acorn_grouped']==g, 'abs_error'].values for g in GROUPS]
H, p = kruskal(*arrays)
print(f"\nKruskal-Wallis:  H = {H:.4f},  p = {p:.4g}")
print("→", "SIGNIFICANT disparity (p < 0.05)" if p < 0.05
          else "No significant disparity (p ≥ 0.05)")


In [ ]:
# Post-hoc: which groups differ? (Dunn with Bonferroni)
if p < 0.05:
    try:
        import scikit_posthocs as sp
        dunn = sp.posthoc_dunn(final, val_col='abs_error',
                               group_col='Acorn_grouped', p_adjust='bonferroni')
        print("Dunn post-hoc (Bonferroni-adjusted p-values):")
        print(dunn.reindex(index=GROUPS, columns=GROUPS).round(4).to_string())
    except ImportError:
        from scipy.stats import mannwhitneyu
        from itertools import combinations
        print("Pairwise Mann-Whitney U (Bonferroni-adjusted):")
        pairs = list(combinations(GROUPS, 2))
        for g1, g2 in pairs:
            a1 = final.loc[final['Acorn_grouped']==g1, 'abs_error']
            a2 = final.loc[final['Acorn_grouped']==g2, 'abs_error']
            _, pv = mannwhitneyu(a1, a2, alternative='two-sided')
            print(f"  {g1:<12} vs {g2:<12}  p = {min(pv*len(pairs),1.0):.4g}")
else:
    print("Kruskal-Wallis not significant — no post-hoc testing required.")


In [ ]:
# Visualise the fairness audit
fig, ax = plt.subplots(1, 3, figsize=(16,4.5))
fig.suptitle('Demographic Fairness Audit — errors of ONE hybrid model, split by ACORN group',
             fontsize=13, fontweight='bold')

# MAE
bars = ax[0].bar(audit.index, audit['MAE'], color=[C[g] for g in audit.index],
                 alpha=0.9, edgecolor='white')
ax[0].set_title(f'MAE by group  (Kruskal-Wallis p = {p:.3g})')
ax[0].set_ylabel('MAE (kWh/h)')
for b_, v in zip(bars, audit['MAE']):
    ax[0].text(b_.get_x()+b_.get_width()/2, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
if p < 0.05:
    ax[0].text(0.5, 0.94, '* significant', transform=ax[0].transAxes,
               ha='center', color='red', fontweight='bold')

# RMSE
bars = ax[1].bar(audit.index, audit['RMSE'], color=[C[g] for g in audit.index],
                 alpha=0.9, edgecolor='white')
ax[1].set_title('RMSE by group'); ax[1].set_ylabel('RMSE (kWh/h)')
for b_, v in zip(bars, audit['RMSE']):
    ax[1].text(b_.get_x()+b_.get_width()/2, v+0.002, f'{v:.4f}', ha='center', fontsize=9)

# error distributions
ax[2].boxplot([final.loc[final['Acorn_grouped']==g,'abs_error'] for g in GROUPS],
              labels=GROUPS, showfliers=False, patch_artist=True,
              boxprops=dict(facecolor='#EAF2F8'), medianprops=dict(color='black'))
ax[2].set_title('Absolute error distribution'); ax[2].set_ylabel('|error| (kWh/h)')

for a_ in ax: a_.tick_params(axis='x', rotation=10)
plt.tight_layout(); plt.savefig('fig_fairness.png', dpi=150, bbox_inches='tight'); plt.show()


## 11. Results Summary

In [ ]:
print("="*64)
print("  RESULTS SUMMARY")
print("="*64)

print("\nRQ1 — Forecasting accuracy")
for _, r in results.iterrows():
    print(f"   {r['Model']:<9} MAE={r['MAE']:.4f}   RMSE={r['RMSE']:.4f}")
best = results.loc[results['MAE'].idxmin(), 'Model']
print(f"   Best model: {best}")
print(f"   Ridge weights — Prophet {meta.coef_[0]:.3f}, LSTM {meta.coef_[1]:.3f}")

print("\nRQ3 — SHAP attribution (hybrid meta-learner)")
for f, v in zip(FNAMES, np.abs(sv).mean(axis=0)):
    print(f"   {f:<20} mean|SHAP| = {v:.4f}")

print("\nRQ2 — Fairness audit (single hybrid model)")
for g in GROUPS:
    r = audit.loc[g]
    print(f"   {g:<12} MAE={r['MAE']:.4f}  RMSE={r['RMSE']:.4f}  "
          f"(n={int(r['Households'])} households, {int(r['N predictions']):,} predictions)")
print(f"   Kruskal-Wallis: H={H:.3f}, p={p:.4g} "
      f"({'SIGNIFICANT' if p<0.05 else 'not significant'})")

print("\nFigures saved:")
for f in ['fig_eda.png','fig_prophet.png','fig_models.png','fig_shap_bar.png',
          'fig_shap_beeswarm.png','fig_shap_by_group.png','fig_fairness.png']:
    print("  ", f)
print("="*64)


---
## Appendix — Data schema

**informations_households.csv**

| Column | Description |
|---|---|
| LCLid | Household ID (e.g. MAC000002) |
| stdorToU | Tariff: Std or ToU |
| Acorn | ACORN class (ACORN-A … ACORN-Q) |
| Acorn_grouped | **Affluent / Comfortable / Adversity** — the fairness subgroups |
| file | Block file the household belongs to |

**halfhourly_dataset/block_N.csv**

| Column | Description |
|---|---|
| LCLid | Household ID (joins to the file above) |
| tstp | Half-hourly timestamp |
| energy(kWh/hh) | Consumption, kWh per half hour ('Null' when missing) |

**Source:** UK Power Networks (2014). *SmartMeter Energy Consumption Data in London Households*.
London Datastore. https://data.london.gov.uk/dataset/smartmeter-energy-use-data-in-london-households
